In [ ]:
# 1. Import libraries
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.preprocessing import image
# 2. Set dataset path
DATA_DIR = os.path.expanduser("~/Desktop/BIA39_LPIV_Mock/cats_and_dogs_filtered")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "validation")
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
# 3. Load training and validation data
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_ds = image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)
class_names = train_ds.class_names
print("Classes:", class_names)
# 4. Preprocess images
train_ds = train_ds.map(
    lambda x, y: (preprocess_input(x), y)
).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(
    lambda x, y: (preprocess_input(x), y)
).prefetch(tf.data.AUTOTUNE)
# 5. Load pretrained MobileNetV2
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(160, 160, 3)
)
# Freeze pretrained layers
base_model.trainable = False
# 6. Build transfer learning model
model = models.Sequential([
    layers.Input(shape=(160, 160, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])
# 7. Compile model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


# 8. Train model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)


# 9. Evaluate model
loss, accuracy = model.evaluate(val_ds)

print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy * 100, "%")

# 10. Plot accuracy
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


# 11. Plot loss
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


# 12. Predict a sample image
sample_path = os.path.join(
    VAL_DIR, class_names[0],
    os.listdir(os.path.join(VAL_DIR, class_names[0]))[0]
)

img = image.load_img(sample_path, target_size=IMG_SIZE)
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

prediction = model.predict(img_array, verbose=0)[0][0]

predicted_class = class_names[int(prediction >= 0.5)]

plt.imshow(img)
plt.title("Predicted: " + predicted_class)
plt.axis("off")
plt.show()

print("Predicted Class:", predicted_class)
print("Prediction Score:", round(float(prediction), 4))


# 13. Save model
model.save("animal_classifier.keras")
print("Model saved successfully!")
